In [1]:
import time

import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_rows", None)  # or a specific number
pd.set_option("display.max_columns", None)  # to show all columns
pd.set_option("display.expand_frame_repr", False)  # to allow wider DataFrame display

In [2]:
features = pd.read_csv("features/features.csv", index_col='match_id')

In [3]:
nafeat = features.isna().any()
print(*list(nafeat[nafeat].index), sep=", ")

first_blood_time, first_blood_team, first_blood_player1, first_blood_player2, radiant_bottle_time, radiant_courier_time, radiant_flying_courier_time, radiant_first_ward_time, dire_bottle_time, dire_courier_time, dire_flying_courier_time, dire_first_ward_time


In [13]:
target_col = "radiant_win"
features_to_remove = [
    "duration",
    "tower_status_radiant",
    "tower_status_dire",
    "barracks_status_dire",
    "barracks_status_radiant",
]
y = features[target_col].copy()
X = features.drop(features_to_remove + [target_col], axis=1)
X = X.fillna(0)

In [14]:
scaler = StandardScaler()

In [6]:
def run_gradient_boosting(X, y, n_estimators=30):
    kf = KFold(n_splits=5, shuffle=True, random_state=241)
    score, elapsed_time = [], []
    for train_ind, test_ind in kf.split(X, y):
        start = time.time()
        gbc = GradientBoostingClassifier(n_estimators=n_estimators, random_state=241)
        gbc.fit(X[train_ind], y[train_ind])
        y_pred_proba = gbc.predict_proba(X[test_ind])[:, 1]
        auc_roc = roc_auc_score(y_true=y[test_ind], y_score=y_pred_proba)
        elapsed_time.append(int(time.time() - start))
        score.append(auc_roc)
    return np.array(score).mean(), np.array(score).std(), np.array(elapsed_time).mean()

In [ ]:
for n_estimators in [10, 20, 30, 50]:
    score_mean, score_std, elapsed_time_mean = run_gradient_boosting(X, y.to_numpy(), n_estimators)
    print(f"n_estimators={n_estimators} | AUC-ROC (mean, std): {score_mean}, {score_std} | mean time: {elapsed_time_mean} sec")

n_estimators=10 | AUC-ROC (mean, std): 0.6643877206345741, 0.004902984301311474 | mean time: 9.0 sec
n_estimators=20 | AUC-ROC (mean, std): 0.6828529377017781, 0.005006424012731842 | mean time: 17.6 sec
n_estimators=30 | AUC-ROC (mean, std): 0.6894967456506673, 0.004528912040751085 | mean time: 26.0 sec
n_estimators=50 | AUC-ROC (mean, std): 0.6974540046869849, 0.003911541855296437 | mean time: 43.8 sec


KeyboardInterrupt: 

In [8]:
def run_logistic_regression(X, y, C=1.0):
    kf = KFold(n_splits=5, shuffle=True, random_state=241)
    score, elapsed_time = [], []
    for train_ind, test_ind in kf.split(X, y):
        start = time.time()
        gbc = LogisticRegression(
            random_state=241,
            penalty="l2",
            C=C,
            max_iter=1000
        )
        gbc.fit(X[train_ind], y[train_ind])
        y_pred_proba = gbc.predict_proba(X[test_ind])[:, 1]
        auc_roc = roc_auc_score(y_true=y[test_ind], y_score=y_pred_proba)
        elapsed_time.append(int(time.time() - start))
        score.append(auc_roc)
    return np.array(score).mean(), np.array(score).std(), np.array(elapsed_time).mean()

In [ ]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_mean = run_logistic_regression(scaler.fit_transform(X), y.to_numpy(), C)
    print(f"C={C} | AUC-ROC (mean, std): {score_mean}, {score_std} | mean time: {elapsed_time_mean} sec")

KeyboardInterrupt: 

In [15]:
X_without_cat = X.drop(
    [
        "lobby_type",
        "r1_hero",
        "r2_hero",
        "r3_hero",
        "r4_hero",
        "r5_hero",
        "d1_hero",
        "d2_hero",
        "d3_hero",
        "d4_hero",
        "d5_hero",
    ], axis=1
)

In [16]:
for C in [1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4]:
    score_mean, score_std, elapsed_time_mean = run_logistic_regression(scaler.fit_transform(X_without_cat), y.to_numpy(), C)
    print(f"C={C} | AUC-ROC (mean, std): {score_mean}, {score_std} | mean time: {elapsed_time_mean} sec")

C=0.0001 | AUC-ROC (mean, std): 0.7112511718603407, 0.0027501847452925466 | mean time: 0.0 sec
C=0.001 | AUC-ROC (mean, std): 0.716237423266761, 0.002796166592407154 | mean time: 0.0 sec
C=0.01 | AUC-ROC (mean, std): 0.7164019187383468, 0.002852507719751197 | mean time: 0.0 sec
C=0.1 | AUC-ROC (mean, std): 0.7163752851856058, 0.0028571888579017194 | mean time: 0.0 sec
C=1.0 | AUC-ROC (mean, std): 0.7163706150683132, 0.0028590503106888923 | mean time: 0.0 sec
C=10.0 | AUC-ROC (mean, std): 0.71637042433271, 0.0028594570231934707 | mean time: 0.0 sec
C=100.0 | AUC-ROC (mean, std): 0.7163703416482603, 0.0028594729921121646 | mean time: 0.4 sec
C=1000.0 | AUC-ROC (mean, std): 0.7163703268178466, 0.002859477946442573 | mean time: 0.0 sec
C=10000.0 | AUC-ROC (mean, std): 0.7163703310575975, 0.002859473019188842 | mean time: 0.0 sec


In [17]:
unique_heros = (
    set(features["r1_hero"].unique())
    | set(features["r2_hero"].unique())
    | set(features["r3_hero"].unique())
    | set(features["r4_hero"].unique())
    | set(features["r5_hero"].unique())
    | set(features["d1_hero"].unique())
    | set(features["d2_hero"].unique())
    | set(features["d3_hero"].unique())
    | set(features["d4_hero"].unique())
    | set(features["d5_hero"].unique())
)

In [18]:
print(f"Number of heroes in game: {max(unique_heros)}")
print(f"Number of unique heroes in dataset: {len(unique_heros)}")

Number of heroes in game: 112
Number of unique heroes in dataset: 108


In [19]:
X_pick = np.zeros((len(features), max(unique_heros)))
for i, match_id in enumerate(features.index):
    for p in range(1, 6):
        X_pick[i, int(features.at[match_id, f"r{p}_hero"])-1]  = 1
        X_pick[i, int(features.at[match_id, f"d{p}_hero"])-1]  = -1

In [20]:
X_pick = pd.DataFrame(X_pick, index=features.index, columns=[f"hero_{i+1}" for i in range(max(unique_heros))])

In [21]:
X_encoded_cat = pd.concat([X_without_cat, X_pick], axis=1)

In [23]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_mean = run_logistic_regression(scaler.fit_transform(X_encoded_cat), y.to_numpy(), C)
    print(f"C={C} | AUC-ROC (mean, std): {score_mean}, {score_std} | mean time: {elapsed_time_mean} sec")

C=0.001 | AUC-ROC (mean, std): 0.7516756856267701, 0.0023186930019418726 | mean time: 0.0 sec
C=0.01 | AUC-ROC (mean, std): 0.7519413309868662, 0.002110656617525964 | mean time: 1.0 sec
C=0.1 | AUC-ROC (mean, std): 0.7519175879925453, 0.0020540620525126042 | mean time: 1.0 sec
C=1 | AUC-ROC (mean, std): 0.751911697953303, 0.0020480734858767976 | mean time: 1.0 sec
C=10 | AUC-ROC (mean, std): 0.7518994488672304, 0.002058471327120526 | mean time: 0.6 sec
C=100 | AUC-ROC (mean, std): 0.7518994022807906, 0.0020584783045041527 | mean time: 0.8 sec
C=1000 | AUC-ROC (mean, std): 0.751899374723686, 0.0020584433697296654 | mean time: 1.0 sec
C=10000 | AUC-ROC (mean, std): 0.7518993789613434, 0.002058453132259211 | mean time: 1.0 sec
